# A6 — XGBoost gbtree (Fair comparison vs current MLflow champion)

This notebook trains only XGBoost_gbtree and ensures the comparison against the current MLflow champion is **fair** by:
- downloading the champion test set artifacts from MLflow (no re-running champion),
- removing any overlap between our training set and the champion test set,
- tuning/training XGBoost on the filtered train set,
- evaluating on the champion test set (paired comparison).

### Imports

In [19]:
import os
import json
import time
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from os import getenv

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, make_scorer

from sklearn.model_selection import RepeatedStratifiedKFold

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

### Paths + load data

In [3]:
SCORES_PATH   = Path("../../data/processed/20190108_scores_and_weak_links.csv")
FEATURES_PATH = Path("../../data/processed/AimoScore_WeakLink_big_scores.csv")

scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)

print("scores:", scores.shape)
print("big:", big.shape)
display(scores.head(2))
display(big.head(2))

scores: (2096, 17)
big: (2094, 43)


,ID,Date,SCORE,ForwardHead,LeftArmFallForward,RightArmFallForward,LeftShoulderElevation,RightShoulderElevation,ExcessiveForwardLean,LeftAsymmetricalWeightShift,RightAsymmetricalWeightShift,LeftKneeMovesInward,RightKneeMovesInward,LeftKneeMovesOutward,RightKneeMovesOutward,LeftHeelRises,RightHeelRises
0,cf0e95ab-d6ec-475a-9cc0-c9444b8dd080.Kinect,2018-07-19,0.612532,0.31,1.34,1.15,0.33,0.49,0.00,0.0,0.08,0.0,0.0,0.0,0.77,0.0,0.0
1,6210cc6d-575d-4e9f-bab1-4fe1da2a30a3.Kinect,2018-07-30,0.388809,1.49,1.42,1.39,0.00,0.00,1.08,0.1,0.00,0.0,0.0,0.0,0.40,0.0,0.0


,AimoScore,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,...,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation,No_1_Time_Deviation,No_2_Time_Deviation,EstimatedScore,ID
0,0.323667,0.538020,0.815878,0.346724,0.382114,0.302248,0.947872,0.275945,0.52176,0.457198,...,0.656624,0.642276,0.552846,0.648972,0.578192,0.560019,0.821616,0.818747,0.209947,0003cdcc-86ed-494a-a3b5-90d09e96e06b.Kinect
1,0.323699,0.443807,0.306552,0.823529,0.188905,0.497370,0.140124,0.664275,0.52176,0.729316,...,0.721186,0.826399,0.805356,0.848876,0.889048,0.816834,0.307987,0.248207,0.457198,003115c4-bdb8-491c-b571-8fcebdecf8ed.Kinect


### MLflow connection

In [4]:
# Adjust if your env is elsewhere
load_dotenv("../../../../../env")

tracking_uri = getenv("MLFLOW_TRACKING_URI")
print("MLFLOW_TRACKING_URI =", tracking_uri)

if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)
else:
    raise RuntimeError("MLFLOW_TRACKING_URI not set. Fix env path/variable.")

EXPERIMENT_NAME = "A6_XGBoost_gbtree_fair"
mlflow.set_experiment(EXPERIMENT_NAME)

client = MlflowClient()
print("Tracking:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

2026/03/07 10:19:31 INFO mlflow.tracking.fluent: Experiment with name 'A6_XGBoost_gbtree_fair' does not exist. Creating a new experiment.


MLFLOW_TRACKING_URI = https://172.27.62.170:5001/
Tracking: https://172.27.62.170:5001/
Experiment: A6_XGBoost_gbtree_fair


### Build WeakestLink labels (drop 5 + drop ties) and merge

In [29]:
# =========================
# A5b preprocessing EXACT
# =========================

DROP_WEAK_LINKS = {
    "LeftHeelRises",
    "RightHeelRises",
    "LeftKneeMovesInward",
    "RightAsymmetricalWeightShift",
}
DROP_TIES = True  # A5b

# 1) Build indicator cols from scores
NON_INDICATORS = {"ID", "Date", "SCORE"}
indicator_cols = [c for c in scores.columns if c not in NON_INDICATORS]
indicator_cols = [c for c in indicator_cols if c not in DROP_WEAK_LINKS]

scores[indicator_cols] = scores[indicator_cols].apply(pd.to_numeric, errors="coerce")

# 2) Tie detection + WeakestLink
row_max = scores[indicator_cols].max(axis=1)
scores["HasTie"] = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)
scores["WeakestLink"] = scores[indicator_cols].idxmax(axis=1)

labels = scores[["ID", "WeakestLink", "HasTie"]].dropna()

# 3) Merge with big
data = big.merge(labels, on="ID", how="inner")
print("Merged:", data.shape, "| tie rows:", int(data["HasTie"].sum()))

# 4) Drop ties
if DROP_TIES:
    data = data.loc[~data["HasTie"]].copy()
    print("After drop ties:", data.shape)

# 5) y
y = data["WeakestLink"].astype(str).reset_index(drop=True)

# 6) A5b feature_cols: Angle + NASM only, sorted
angle_cols = [c for c in data.columns if c.startswith("No_") and c.endswith("_Angle_Deviation")]
nasm_cols  = [c for c in data.columns if c.startswith("No_") and c.endswith("_NASM_Deviation")]
feature_cols = sorted(angle_cols) + sorted(nasm_cols)

X = data[feature_cols].apply(pd.to_numeric, errors="coerce")

# 7) A5b NaN policy: drop rows with any NaN in X
mask = X.notna().all(axis=1)
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

print("Final X:", X.shape, "Classes:", y.nunique(), "Min class:", int(y.value_counts().min()))
display(y.value_counts().head(20))

Merged: (2094, 45) | tie rows: 368
After drop ties: (1726, 45)


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

### Split + CV (our notebook split)

In [24]:
RANDOM_SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

min_class = int(y_train.value_counts().min())
cv_splits = 5
cv = RepeatedStratifiedKFold(n_splits=cv_splits, n_repeats=3, random_state=RANDOM_SEED)
print("X_train:", X_train.shape, "X_test:", X_test.shape, "cv_splits:", cv_splits)

X_train: (1380, 38) X_test: (346, 38) cv_splits: 5


### Label encode y for XGBoost

In [25]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("Encoded classes:", list(le.classes_))
print("n_classes:", len(le.classes_))

Encoded classes: ['ExcessiveForwardLean', 'ForwardHead', 'LeftArmFallForward', 'LeftAsymmetricalWeightShift', 'LeftKneeMovesOutward', 'LeftShoulderElevation', 'RightArmFallForward', 'RightKneeMovesInward', 'RightKneeMovesOutward', 'RightShoulderElevation']
n_classes: 10


### Download champion artifacts (X_test + outputs) from MLflow

In [26]:
REGISTERED_MODEL_NAME = "weakest_link_model"

mv = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "champion")
champ_run_id = mv.run_id
print("Champion run_id:", champ_run_id)

artifact_candidates_X = [
    "X_test_inputs.csv",
    "mlflow_champion_artifacts/X_test_inputs.csv",
    "mlflow_weightedf1_artifacts/X_test_inputs.csv",
]
artifact_candidates_Y = [
    "test_outputs_predictions.csv",
    "mlflow_champion_artifacts/test_outputs_predictions.csv",
    "mlflow_weightedf1_artifacts/test_outputs_predictions.csv",
]

champ_X_test = None
for ap in artifact_candidates_X:
    try:
        p = mlflow.artifacts.download_artifacts(run_id=champ_run_id, artifact_path=ap)
        if os.path.exists(p):
            champ_X_test = pd.read_csv(p)
            print("Downloaded champion X_test from:", ap)
            break
    except Exception:
        pass

champ_pred_df = None
for ap in artifact_candidates_Y:
    try:
        p = mlflow.artifacts.download_artifacts(run_id=champ_run_id, artifact_path=ap)
        if os.path.exists(p):
            champ_pred_df = pd.read_csv(p)
            print("Downloaded champion outputs from:", ap)
            break
    except Exception:
        pass

if champ_X_test is None or champ_pred_df is None:
    raise FileNotFoundError("Could not download champion X_test or outputs. Check artifact paths in MLflow UI.")

if "y_true" not in champ_pred_df.columns or "y_pred" not in champ_pred_df.columns:
    raise ValueError("Champion outputs must contain columns y_true and y_pred.")

print("champ_X_test:", champ_X_test.shape)
print("champ_pred_df rows:", len(champ_pred_df))
display(champ_pred_df.head(3))

Champion run_id: 8f2f9018adb94b558ef66824887b0e3d
Downloaded champion X_test from: X_test_inputs.csv
Downloaded champion outputs from: test_outputs_predictions.csv
champ_X_test: (345, 38)
champ_pred_df rows: 345


,y_true,y_pred,proba_ExcessiveForwardLean,proba_ForwardHead,proba_LeftArmFallForward,proba_LeftAsymmetricalWeightShift,proba_LeftKneeMovesOutward,proba_LeftShoulderElevation,proba_RightArmFallForward,proba_RightKneeMovesInward,proba_RightKneeMovesOutward,proba_RightShoulderElevation
0,ForwardHead,ForwardHead,0.000003,0.991244,0.000011,4.526746e-07,3.079165e-07,0.000001,0.008732,4.737497e-07,4.427818e-07,0.000008
1,LeftArmFallForward,RightArmFallForward,0.000076,0.000025,0.348966,1.527699e-05,3.471003e-06,0.000044,0.650453,1.849147e-05,6.468604e-06,0.000393
2,RightKneeMovesInward,RightShoulderElevation,0.000014,0.000010,0.000271,3.945845e-05,9.720722e-05,0.000123,0.001262,1.010446e-03,7.164204e-05,0.997100


### Align columns + compute overlap and create FAIR training set

In [27]:
def row_fingerprint(df: pd.DataFrame) -> pd.Series:
    df2 = df.copy()
    for c in df2.columns:
        if pd.api.types.is_float_dtype(df2[c]):
            df2[c] = df2[c].round(10)
    return df2.astype(str).agg("|".join, axis=1).apply(lambda s: hashlib.md5(s.encode()).hexdigest())

# Align champion X_test columns to our training columns
champ_X_test = champ_X_test.reindex(columns=X_train.columns)

fp_ch = set(row_fingerprint(champ_X_test))
fp_train = row_fingerprint(X_train)
keep_mask = ~fp_train.isin(fp_ch)

overlap = int((~keep_mask).sum())
print("Champion test rows:", len(champ_X_test))
print("Overlap with OUR X_train:", overlap)
print("Overlap %:", overlap / len(X_train))

X_train_fair = X_train.loc[keep_mask].copy()
y_train_fair_enc = y_train_enc[keep_mask.values]

print("Original X_train:", X_train.shape)
print("Fair X_train:", X_train_fair.shape)

Champion test rows: 345
Overlap with OUR X_train: 49
Overlap %: 0.035507246376811595
Original X_train: (1380, 38)
Fair X_train: (1331, 38)


### Tune XGBoost_gbtree on FAIR training set (CV weighted F1)

In [28]:
# scorer on encoded labels (decode inside)
def weighted_f1_on_encoded(y_true_enc, y_pred_enc):
    yt = le.inverse_transform(np.asarray(y_true_enc).astype(int))
    yp = le.inverse_transform(np.asarray(y_pred_enc).astype(int))
    return f1_score(yt, yp, average="weighted", zero_division=0)

f1_weighted_encoded = make_scorer(weighted_f1_on_encoded)

FAST_MODE = True
N_ITER = 15  # start small; increase for competition

xgb_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=len(le.classes_),
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=RANDOM_SEED,
        n_estimators=600 if FAST_MODE else 1500,
        verbosity=0,
    ))
])

param_xgb_tree = {
    "xgb__max_depth": [3, 4, 5, 6, 8],
    "xgb__learning_rate": np.logspace(-2.1, -0.7, 18),
    "xgb__subsample": [0.6, 0.8, 1.0],
    "xgb__colsample_bytree": [0.6, 0.8, 1.0],
    "xgb__min_child_weight": [1, 2, 5, 10],
    "xgb__gamma": [0.0, 0.1, 0.3, 1.0],
    "xgb__reg_alpha": np.logspace(-4, -0.5, 10),
    "xgb__reg_lambda": np.logspace(-2, 1, 10),
}

search = RandomizedSearchCV(
    estimator=xgb_tree,
    param_distributions=param_xgb_tree,
    n_iter=N_ITER,
    scoring=f1_weighted_encoded,
    cv=cv,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=1,
    refit=True,
    error_score=0.0,
)

t0 = time.perf_counter()
search.fit(X_train_fair, y_train_fair_enc)
t1 = time.perf_counter()

best_xgb_fair = search.best_estimator_
print("\n==== XGBoost_gbtree (FAIR) ====")
print("CV weighted_f1:", float(search.best_score_))
print("Best params:", search.best_params_)
print("Seconds:", float(t1 - t0))

Fitting 15 folds for each of 15 candidates, totalling 225 fits

==== XGBoost_gbtree (FAIR) ====
CV weighted_f1: 0.6228163813466991
Best params: {'xgb__subsample': 0.8, 'xgb__reg_lambda': np.float64(4.6415888336127775), 'xgb__reg_alpha': np.float64(0.00024484367468222265), 'xgb__min_child_weight': 2, 'xgb__max_depth': 8, 'xgb__learning_rate': np.float64(0.014030037231905735), 'xgb__gamma': 0.3, 'xgb__colsample_bytree': 0.8}
Seconds: 513.0724107999995


### Evaluate FAIR model on champion test set + compare to champion

In [11]:
y_true_ch = champ_pred_df["y_true"].astype(str).reset_index(drop=True).values
y_pred_ch = champ_pred_df["y_pred"].astype(str).reset_index(drop=True).values

pred_cur_fair = le.inverse_transform(best_xgb_fair.predict(champ_X_test).astype(int))

wf1_cur = f1_score(y_true_ch, pred_cur_fair, average="weighted", zero_division=0)
wf1_ch  = f1_score(y_true_ch, y_pred_ch, average="weighted", zero_division=0)

acc_cur = accuracy_score(y_true_ch, pred_cur_fair)
acc_ch  = accuracy_score(y_true_ch, y_pred_ch)

print("FAIR eval on champion X_test (same 345 rows):")
print(" Current weighted_f1:", float(wf1_cur), " accuracy:", float(acc_cur))
print(" Champion weighted_f1:", float(wf1_ch), " accuracy:", float(acc_ch))
print(" Delta weighted_f1:", float(wf1_cur - wf1_ch))

FAIR eval on champion X_test (same 345 rows):
 Current weighted_f1: 0.8362092361277073  accuracy: 0.8492753623188406
 Champion weighted_f1: 0.654381440617033  accuracy: 0.663768115942029
 Delta weighted_f1: 0.1818277955106743


### Log run to MLflow (no registry promotion here)

In [12]:
artifact_dir = "mlflow_fair_xgb_artifacts"
os.makedirs(artifact_dir, exist_ok=True)

# Save fair-eval predictions
pred_df = pd.DataFrame({
    "y_true": y_true_ch,
    "y_pred_current": pred_cur_fair,
    "y_pred_champion": y_pred_ch,
})
pred_path = os.path.join(artifact_dir, "paired_eval_predictions.csv")
pred_df.to_csv(pred_path, index=False)

with mlflow.start_run(run_name="XGBoost_gbtree_FAIR") as run:
    mlflow.set_tag("task", "classification")
    mlflow.set_tag("project_component", "weakest_link")
    mlflow.set_tag("model_key", "XGBoost_gbtree_fair")
    mlflow.set_tag("champion_run_id", champ_run_id)

    mlflow.log_param("n_iter_search", N_ITER)
    mlflow.log_param("cv_splits", cv_splits)
    mlflow.log_param("removed_overlap_rows", overlap)
    mlflow.log_param("drop_weak_links", json.dumps(sorted(list(DROP_WEAK_LINKS))))
    mlflow.log_params({k: (float(v) if isinstance(v, (np.floating,)) else v) for k, v in search.best_params_.items()})

    mlflow.log_metric("fair_champion_test_weighted_f1_current", float(wf1_cur))
    mlflow.log_metric("fair_champion_test_weighted_f1_champion", float(wf1_ch))
    mlflow.log_metric("fair_champion_test_weighted_f1_delta", float(wf1_cur - wf1_ch))

    mlflow.log_metric("fair_champion_test_accuracy_current", float(acc_cur))
    mlflow.log_metric("fair_champion_test_accuracy_champion", float(acc_ch))

    mlflow.log_artifact(pred_path)

print("Logged fair comparison run to MLflow.")

🏃 View run XGBoost_gbtree_FAIR at: https://172.27.62.170:5001/#/experiments/14/runs/443b484cc68d4301a21b956b818cb1e6
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/14
Logged fair comparison run to MLflow.
